In [1]:
# Import modules
import geopandas as gpd
import numpy as np
import pandas as pd
import os
from os.path import join
import glob

In [2]:
# Set your working directory
os.chdir("../..")
working_dir = os.getcwd()

In [3]:
print(working_dir)

/home/user/These/cordex_htws_cc3d


In [4]:
# Load data
shapefile_path = join(working_dir,"Data/interactive-atlas/IPCC-WGI-reference-regions-v4_shapefile/IPCC_WG1_reference_regions_v4.gpkg") # downloaded from https://github.com/IPCC-WG1/Atlas/blob/main/reference-regions/IPCC-WGI-reference-regions-v4_shapefile.zip and added regions area in QGIS
datadir = join(working_dir,"Data/interactive-atlas/dataset-aggregated-regionally") # downloaded from https://github.com/IPCC-WG1/Atlas/tree/main/datasets-aggregated-regionally/data/CMIP6/CMIP6_tas_landsea
csv_filelist = glob.glob(join(datadir,'*historical*.csv')) # select csv files of historical runs

In [5]:
# Compute area of the 4 european zone and their fraction of sum of the 4 areas
db_ref_regions = gpd.read_file(shapefile_path)
eurocordex_regions = ['NEU','WCE','EEU','MED'] # the 4 regions overlapping at least 80% with the EUROCORDEX region, according to IPPC : https://github.com/IPCC-WG1/Atlas/blob/main/datasets-aggregated-regionally/README.md
db_europe = db_ref_regions[db_ref_regions["Acronym"].isin(eurocordex_regions)]
europe_area = db_europe["area"].sum()
weight_dict = {}
for region in eurocordex_regions :
    weight_dict[region] = db_ref_regions[db_ref_regions["Acronym"]==region]['area'].values[0]/europe_area

In [7]:
db_ref_regions

,Continent,Type,Name,Acronym,area,%area,geometry
0,POLAR,Land,Greenland/Iceland,GIC,4.773859e+10,0.93586,"MULTIPOLYGON (((-10 62, -10.4375 62, -10.875 6..."
1,NORTH-AMERICA,Land,N.W.North-America,NWN,7.508144e+10,1.47189,"MULTIPOLYGON (((-105 50, -105.4386 50, -105.87..."
2,NORTH-AMERICA,Land,N.E.North-America,NEN,7.664688e+10,1.50258,"MULTIPOLYGON (((-50 50, -50.44 50, -50.88 50, ..."
3,NORTH-AMERICA,Land,W.North-America,WNA,3.138781e+10,0.61532,"MULTIPOLYGON (((-130 50, -129.5614 50, -129.12..."
4,NORTH-AMERICA,Land,C.North-America,CNA,2.931387e+10,0.57467,"MULTIPOLYGON (((-90 50, -90 49.5614, -90 49.12..."
5,NORTH-AMERICA,Land,E.North-America,ENA,5.686132e+10,1.11471,"MULTIPOLYGON (((-70 25, -70.43478 25, -70.8695..."
6,CENTRAL-AMERICA,Land,N.Central-America,NCA,3.161337e+10,0.61975,"MULTIPOLYGON (((-90 25, -90.37179 24.76923, -9..."
7,CENTRAL-AMERICA,Land,S.Central-America,SCA,3.851208e+10,0.75499,"MULTIPOLYGON (((-75 12, -75.28 11.67333, -75.5..."
8,CENTRAL-AMERICA,Land-Ocean,Caribbean,CAR,3.032671e+10,0.59452,"MULTIPOLYGON (((-75 12, -75.32609 12.28261, -7..."
9,SOUTH-AMERICA,Land,N.W.South-America,NWS,3.121109e+10,0.61186,"MULTIPOLYGON (((-75 12, -74.57143 12, -74.1428..."


In [8]:
# Compute Global Warming Level and Regional Warming Level of the period 1986-2005
db_avg = pd.DataFrame(index=range(len(csv_filelist)),columns=["model","world_avg","eurocordex_avg"])
for i in range(len(csv_filelist)) :
    with open(csv_filelist[i],'r') as f :
        model = f.readlines()[4][8:-1] # Get model name
    db_avg.loc[i,"model"] = model
    db_temp = pd.read_csv(csv_filelist[i],sep=",",index_col=0,header=15)
    db_temp_ref = db_temp.loc["1850-01":"1899-12",['NEU','WCE','EEU','MED','world']].mean() # Average of the 1850-1899 period
    db_temp = db_temp.loc["1986-01":"2005-12",['NEU','WCE','EEU','MED','world']].mean() # Average of the 1986-2005 period
    db_avg.loc[i,"world_avg"] = db_temp["world"] - db_temp_ref["world"] # Compute Global Warming Level of 1986-2005
    europe_avg = 0
    europe_avg_ref = 0
    for region in eurocordex_regions :
        # Weigh each european region by its area fraction in order to compute the european average temperature (for period 1986-2005 and reference period 1850-1899)
        europe_avg += db_temp[region]*weight_dict[region]
        europe_avg_ref += db_temp_ref[region]*weight_dict[region]

    db_avg.loc[i,"eurocordex_avg"] = europe_avg - europe_avg_ref # Compute Regional Warming Level of the period 1986-2005

In [9]:
# Show dataframe
db_avg

,model,world_avg,eurocordex_avg
0,NorESM2-LM_r1i1p1f1,0.301291,0.241505
1,EC-Earth3-Veg_r1i1p1f1,1.08569,2.167852
2,ACCESS-CM2_r1i1p1f1,0.392181,0.129123
3,EC-Earth3-Veg-LR_r1i1p1f1,0.70588,1.404576
4,FGOALS-g3_r1i1p1f1,0.710552,0.66292
5,HadGEM3-GC31-LL_r1i1p1f3,0.346038,0.173803
6,CESM2_r4i1p1f1,0.6629,0.75702
7,ACCESS-ESM1-5_r1i1p1f1,0.459394,0.546566
8,EC-Earth3_r1i1p1f1,0.446919,0.023128
9,TaiESM1_r1i1p1f1,0.157973,-0.144117


In [8]:
print("mean global warming: ", db_avg["world_avg"].mean())
print("median global warming: ",db_avg["world_avg"].quantile(0.5))

mean global warming:  0.5927290476190479
median global warming:  0.592606666666665


In [ ]:
print("mean regional warming: ", db_avg["eurocordex_avg"].mean())
print("median regional warming: ",db_avg["eurocordex_avg"].quantile(0.5))

mean regional warming:  0.701918381186525
median regional warming:  0.701918381186525


In [10]:
# Litterature shows the GWL of 1986-2005 is 0.61°C so we use it to correct the computed Regional Warming Level
expected_global_warming = 0.61
european_warming = db_avg["eurocordex_avg"].quantile(0.5)*expected_global_warming/db_avg["world_avg"].quantile(0.5)
print(f"European warming 1986-2005: {european_warming}°C")

European warming 1986-2005: 0.722520073782129°C
